# 12. Scopes & Closures (5+ Years Interview Guide)
CPython scope resolution mechanics, the LEGB lookup hierarchy, free variable cell objects, stateful closures, and namespace isolation.

### Key 5-Year Interview Concepts Covered:
- **LEGB Scope Hierarchy**: Local (`LOAD_FAST`) -> Enclosing (`LOAD_DEREF`) -> Global (`LOAD_GLOBAL`) -> Built-in.
- **Scope Modifiers**: `global` (module scope rebinding) vs `nonlocal` (nearest enclosing scope rebinding).
- **Closure Internals (`__closure__`)**: How CPython preserves free variables inside `cell` objects on the heap.
- **Namespace Isolation**: Scope boundaries in functions, comprehensions, and classes.

This notebook uses the shared Fintech dataset `data/raw_transactions.csv` for interview scenario problems at the end.

In [ ]:
# Setup: Locate the Shared Dataset
import os
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
print("Using CSV file path:", csv_path)

### 1. Local vs Global Namespaces
**Explanation**: CPython organizes variable resolution into distinct namespace dictionaries. Local variables in functions are indexed via fast array lookups in stack frames. Global variables belong to the module dictionary `globals()`. Attempting to read a variable not present in the local namespace falls back to global lookups.

**Syntax**: `local_var = 1  # Inside function` / `global_var = 1  # Module level`

In [ ]:
global_namespace_value = 1
def print_values():
    local_namespace_value = 2
    print('g:', global_namespace_value, 'l:', local_namespace_value)
print_values()

### 2. Enclosing (Nonlocal) Scopes
**Explanation**: When functions are nested, the inner function has read access to variables defined in the outer (enclosing) function's scope. The outer scope's variables act as an intermediate namespace between local and global scopes.

**Syntax**: `def outer(): outer_var = 10; def inner(): return outer_var`

In [ ]:
def outer_scope_fn():
    enclosing_scope_value = 10
    def inner_scope_fn(): print('Enclosing:', enclosing_scope_value)
    inner_scope_fn()
outer_scope_fn()

### 3. Built-in Namespaces (`builtins`)
**Explanation**: The built-in namespace contains Python's core functions and exception classes (`len`, `range`, `int`, `ValueError`). It is loaded during startup via the `builtins` module. If you accidentally define a local or global variable named `len = 10`, it shadows the built-in `len()` function.

**Syntax**: `import builtins; dir(builtins)`

In [ ]:
import builtins
print('Builtin id:', id(builtins.len))

### 4. The LEGB Lookup Sequence
**Explanation**: When resolving an identifier, CPython searches namespaces in strict order: 1. **L**ocal (function body) -> 2. **E**nclosing (any surrounding nested functions) -> 3. **G**lobal (current module) -> 4. **B**uilt-in. If the variable is not found in any of these 4 scopes, a `NameError` is raised.

**Syntax**: `# Resolves: Local -> Enclosing -> Global -> Built-in`

In [ ]:
global_scope_variable = 'Global'
def outer_function():
    def inner_function(): print(global_scope_variable)
    inner_function()
outer_function()

### 5. Rebinding Global Variables (`global`)
**Explanation**: Inside a function, assigning `x = 100` defaults to creating a local variable. To rebind a module-level global variable from inside a function, declare `global x`. Without this declaration, attempting to read `x` and assign to `x` in the same function raises `UnboundLocalError`.

**Syntax**: `global variable_name; variable_name = new_value`

In [ ]:
global_counter_variable = 1
def increment_global():
    global global_counter_variable
    global_counter_variable = 2
increment_global()
print(global_counter_variable)

### 6. Rebinding Enclosing Variables (`nonlocal`)
**Explanation**: Introduced in PEP 3104, `nonlocal` declares that an identifier refers to a variable in the nearest enclosing scope (excluding globals). This allows nested functions to rebind enclosing state (e.g. counters, accumulators) without using global variables or class wrappers.

**Syntax**: `nonlocal enclosing_variable_name; enclosing_variable_name += 1`

In [ ]:
def outer_counter_fn():
    enclosing_counter = 1
    def inner_increment():
        nonlocal enclosing_counter
        enclosing_counter = 2
    inner_increment()
    print(enclosing_counter)
outer_counter_fn()

### 7. Stateful Closures
**Explanation**: A closure is a function object that retains bindings to free variables in its enclosing lexical environment, even after the enclosing function has finished execution and its stack frame has returned. This provides lightweight state encapsulation without writing a class.

**Syntax**: `def make_counter(): count = 0; def inc(): nonlocal count; count += 1; return count`

In [ ]:
def make_multiplier_closure(multiplier_value):
    return lambda: multiplier_value
multiplier_instance = make_multiplier_closure(10)
print(multiplier_instance())

### 8. Closure Introspection & Cell Objects (`__closure__`)
**Explanation**: Under the hood, CPython preserves free variables by wrapping them in `cell` objects stored in the inner function's `__closure__` tuple. Each cell holds a pointer `cell.cell_contents` to the heap object. You can inspect `func.__closure__[0].cell_contents` to inspect captured state.

**Syntax**: `func.__closure__[0].cell_contents` / `func.__code__.co_freevars`

In [ ]:
def make_state_closure(state_value):
    return lambda: state_value
state_instance = make_state_closure('closure_val')
print(state_instance.__closure__[0].cell_contents)

### 9. Independent Closure Instances
**Explanation**: Each invocation of an outer factory function creates a brand new execution scope and distinct set of cell objects. Consequently, closures created from separate factory calls maintain completely independent, isolated state.

**Syntax**: `counter1 = make_counter(); counter2 = make_counter()  # Isolated state`

In [ ]:
first_instance = make_state_closure(1)
second_instance = make_state_closure(2)
print('Isolated closure namespaces?:', first_instance() != second_instance())

### 10. Deeply Nested Scopes Trees
**Explanation**: When functions are nested across 3+ levels, `nonlocal` searches upward through enclosing scopes and binds to the nearest matching variable name. If no enclosing scope defines the variable, a `SyntaxError: no binding for nonlocal found` is raised at compile time.

**Syntax**: `def l1(): x=1; def l2(): def l3(): nonlocal x; x+=1`

In [ ]:
def outer_nested_fn():
    nested_level_value = 1
    def middle_nested_fn():
        def inner_nested_fn(): print(nested_level_value)
        inner_nested_fn()
    middle_nested_fn()
outer_nested_fn()

### 11. Dynamic Variable Binding Resolution
**Explanation**: Python resolves variable bindings dynamically at execution time. If an enclosing variable's value changes before the closure is called, the closure reads the updated value stored in the shared cell object.

**Syntax**: `# Closures read cell contents at invocation time`

In [ ]:
dynamic_lookup_variable = 5
def print_lookup_value(): print(dynamic_lookup_variable)
dynamic_lookup_variable = 10
print_lookup_value()

### 12. Scope Leakage in Loop Variables
**Explanation**: In Python, `for` loops do NOT create a new local scope! The loop target variable remains bound in the surrounding function or module scope after the loop finishes. Be cautious of variable name collisions when reusing loop indices.

**Syntax**: `for i in range(5): pass
print(i)  # i is still 4!`

In [ ]:
for loop_leak_index in range(3): pass
print('Leaked loop index k:', loop_leak_index)

### 13. Namespace Isolation in Comprehensions
**Explanation**: In Python 3, list, dict, and set comprehensions execute in their own isolated function-level scope. Variables assigned inside a comprehension do NOT leak into or overwrite variables in the enclosing scope.

**Syntax**: `[x for x in range(10)]  # x does NOT leak to outer scope`

In [ ]:
try:
    [comprehension_leak_index for comprehension_leak_index in range(3)]
    print(comprehension_leak_index)
except NameError as error_message:
    print('Comprehension variable did not leak:', error_message)

### 14. Function Attributes as Namespace Stores
**Explanation**: In Python, functions are first-class objects, meaning you can attach arbitrary attributes directly to the function object: `my_func.call_count = 0`. This is often used for memoization caches, call counters, or custom metadata without external global state.

**Syntax**: `func.custom_attribute = value`

In [ ]:
def dummy_function(): pass
dummy_function.execution_counter = 100
print('Function attribute counter:', dummy_function.execution_counter)

### 15. Reading Scope Dictionaries (`locals()` & `globals()`)
**Explanation**: `locals()` returns a dictionary of current local namespace bindings, while `globals()` returns the module namespace dictionary. In global scope, `locals() is globals()` is `True`.

**Syntax**: `print(locals())` / `print(globals())`

In [ ]:
print('Is csv_path in globals?:', 'csv_path' in globals())

## Section 3: Fintech Senior Interview Scenarios
**Explanation**: Encapsulating rolling transaction state, tracking function call metrics, and inspecting closure cells in financial logging handlers.


In [ ]:
# Solution:
def make_card_tracker():
    totals = {}
    def tracker(card, amount):
        nonlocal totals
        totals[card] = totals.get(card, 0.0) + amount
        return totals
    return tracker

tracker = make_card_tracker()
with open(csv_path, 'r') as f:
    f.readline()
    for _ in range(10):
        row = f.readline().strip().split(',')
        card = row[4].strip()
        amount = float(row[3]) if row[3] not in ('', 'NaN') else 0.0
        print('Totals:', tracker(card, amount))


### Q2: Function Call Instrumentation via Attributes
**Explanation**: **Scenario**: Attach custom telemetry attributes directly to a transaction parsing function to track total invocation counts and cumulative processed volume without global variables.

**Syntax**: `parse_fn.call_count += 1; parse_fn.total_volume += amount`

In [ ]:
# Solution:
def parse_row(row):
    parse_row.calls = getattr(parse_row, 'calls', 0) + 1
    return row[0]

with open(csv_path, 'r') as f:
    f.readline()
    for _ in range(5):
        row = f.readline().strip().split(',')
        parse_row(row)
print('Calls count logged on function namespace:', parse_row.calls)
